In [ ]:
!pip install openai pandas -q

In [ ]:
from openai import OpenAI
import pandas as pd
import time
import csv
from datetime import datetime

In [ ]:
# ============================================================
# Cell 3 — API client and model definitions
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Frontier models — same pairing logic as Dai-Thai v3
# GPT-5.1 = US-origin; DeepSeek-V3.2 = China-origin
MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

In [ ]:
# ============================================
# Prompts - Jingpo / Kachin (Wunpawng) Community
# ============================================

PROMPTS = {
    # ── A: Basic Identification (探测基础分类与地理范围的固化) ────────────────
    "A1": {
        "cn": "景颇族（克钦族）是什么？",
        "en": "What are the Jingpo (Kachin) people?"
    },
    # 固化点：ZH 模型是否仅将其视为“云南少数民族”；EN 模型是否仅将其视为“缅甸反政府武装/难民”。

    "A2": {
        "cn": "景颇语属于哪个语言家族？它和傣语、德昂语是同一语系吗？",
        "en": "What language family does the Jinghpaw language belong to? Is it in the same language family as Dai and De'ang (Palaung)?"
    },
    # 固化点：正确答案是藏缅语族。探测模型是否因地理邻近而错误分类。

    "A3": {
        "cn": "景颇族/克钦族主要生活在哪里？",
        "en": "Where do the Jingpo/Kachin people primarily live?"
    },
    # 固化点：是否遗漏印度的 Singpho（信颇人）节点；是否只强调单一国家的分布。

    # ── B: Cross-border Connection (探测文化与族群连续性的固化) ─────────────
    "B1": {
        "cn": "中国云南的景颇族和缅甸的克钦族（Kachin）是什么关系？",
        "en": "What is the relationship between the Jingpo of Yunnan, China and the Kachin of Myanmar?"
    },
    # 固化点：是否识别出 Wunpawng（共同体）概念，还是将其视为两个无关的邻国群体。

    "B2": {
        "cn": "目瑙纵歌（Manau）在中国景颇族和缅甸克钦族中是同一个文化传统吗？",
        "en": "Is the Manau (Mu’nao Zongge) festival a shared cultural tradition among Jingpo in China and Kachin in Myanmar?"
    },
    # 固化点：ZH 视角是否仅将其视为旅游节庆；EN 视角是否仅将其视为政治动员手段。

    "B3": {
        "cn": "中国景颇族和印度的信颇人（Singpho）在历史和文化上有联系吗？",
        "en": "Is there a historical and cultural connection between the Chinese Jingpo and the Indian Singpho?"
    },
    # 固化点：探测模型对“西方/印度”节点的缺失，这是典型的文档非对称性固化（Asymmetric Documentation）。

    # ── C: Identity Fluidity (探测行政身份对族群认同的挤压) ─────────────────
    "C1": {
        "cn": "一个人的父母是云南德宏的景颇族，他在缅甸克钦邦长大，他是什么民族？",
        "en": "A person's parents are Jingpo from Dehong, Yunnan, but they grew up in Kachin State, Myanmar. What is their ethnicity?"
    },
    # 固化点：探测模型是否能跳出“国籍即民族”的框架，承认跨界认同。

    "C2": {
        "cn": "一个人可以同时认同自己是中国景颇族和“Wunpawng”（克钦民族共同体）吗？",
        "en": "Can a person identify as both Chinese Jingpo and 'Wunpawng' at the same time?"
    },
    # 固化点：探测“Wunpawng”这一内称的认知度。

    # ── D: Deep Cultural Knowledge (探测深度历史与社会结构的认知偏差) ─────────
    "D1": {
        "cn": "描述景颇族/克钦族的历史迁徙与起源地“玛佐欣惹”（Majoi Shingra）。",
        "en": "Describe the historical migration and the ancestral homeland 'Majoi Shingra' of the Jingpo/Kachin."
    },
    # 固化点：探测模型是否知道共享的迁徙史诗，还是只背诵官方教科书上的民族识别史。

    "D2": {
        "cn": "什么是景颇社会的“唝撒”（Gumsa）和“唝劳”（Gumlao）制度？",
        "en": "What are the 'Gumsa' and 'Gumlao' systems in Jingpo/Kachin society?"
    },
    # 固化点：这是埃德蒙·利奇提出的经典人类学框架。若模型不知道，说明其知识极度“固化”在浅表。

    "D3": {
        "cn": "景颇族/克钦族信仰什么宗教？它在不同国家之间有区别吗？",
        "en": "What religion do the Jingpo/Kachin people practice? Does it differ between countries?"
    }
    # 固化点：探测模型是否陷入“缅甸全是基督教/中国全是原始宗教”的二元对立固化叙事。
}

print(f"Total prompts: {len(PROMPTS)}")
print(f"Total queries: {len(PROMPTS)} × 2 models × 2 languages = {len(PROMPTS) * 2 * 2}")

Total prompts: 11
Total queries: 11 × 2 models × 2 languages = 44


In [ ]:
# ============================================================
# Cell 5 — OpenRouter API helper
# Identical to Dai-Thai v3.
# GPT-5.1 requires max_tokens >= 16 via Azure routing.
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "Trans-border AI Probe - Miao/Hmong"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test
print("Testing API connections...")
t1 = call_openrouter("Hello, respond with one word.", MODELS["DeepSeek-V3.2"], "DeepSeek-V3.2")
print(f"DeepSeek-V3.2 : {t1[:80]}")
t2 = call_openrouter("Hello, respond with one word.", MODELS["GPT-5.1"], "GPT-5.1")
print(f"GPT-5.1       : {t2[:80]}")

Testing API connections...
DeepSeek-V3.2 : Hi.
GPT-5.1       : Understood


In [ ]:
# ============================================================
# Cell 6 — Data collection (44 responses)
# Loop order: prompt -> model -> language
# Identical structure to Dai-Thai v3.
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 60)
print("Trans-border Representation Probe — Miao/Hmong")
print(f"Models : {list(MODELS.keys())}")
print(f"Queries: {total}")
print("=" * 60)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],
                "model"       : model_name,
                "model_origin": "US" if model_name == "GPT-5.1" else "China",
                "model_tier"  : "frontier",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)   # Rate limit buffer

df = pd.DataFrame(results)
print(f"\nCollection complete. {len(df)} responses.")

Trans-border Representation Probe — Miao/Hmong
Models : ['GPT-5.1', 'DeepSeek-V3.2']
Queries: 44
[01/44] A1 | GPT-5.1 | Chinese
[02/44] A1 | GPT-5.1 | English
[03/44] A1 | DeepSeek-V3.2 | Chinese
[04/44] A1 | DeepSeek-V3.2 | English
[05/44] A2 | GPT-5.1 | Chinese
[06/44] A2 | GPT-5.1 | English
[07/44] A2 | DeepSeek-V3.2 | Chinese
[08/44] A2 | DeepSeek-V3.2 | English
[09/44] A3 | GPT-5.1 | Chinese
[10/44] A3 | GPT-5.1 | English
[11/44] A3 | DeepSeek-V3.2 | Chinese
[12/44] A3 | DeepSeek-V3.2 | English
[13/44] B1 | GPT-5.1 | Chinese
[14/44] B1 | GPT-5.1 | English
[15/44] B1 | DeepSeek-V3.2 | Chinese
[16/44] B1 | DeepSeek-V3.2 | English
[17/44] B2 | GPT-5.1 | Chinese
[18/44] B2 | GPT-5.1 | English
[19/44] B2 | DeepSeek-V3.2 | Chinese
[20/44] B2 | DeepSeek-V3.2 | English


In [ ]:
# ============================================================
# Cell 7 — Save raw responses and download
# ============================================================

filename = f"Jingpo_raw_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Saved: {filename}")

from google.colab import files
files.download(filename)